# Annotation des taxonomies avec une IA

Ce notebook parcourt les images et les planches sources du dataset local et utilise Qwen3.8-Flash-Next Gyro-S quantifié sur un serveur Strata distant pour proposer des étiquettes à partir des arbres de [`taxonomies.json`](taxonomies.json). Les définitions de chaque valeur sont transmises au modèle. Chaque dimension possède une hiérarchie indépendante : les feuilles deviennent des étiquettes et leur chemin complet est conservé. Le notebook [data_structure.ipynb](data_structure.ipynb) permet de les visualiser. Chaque résultat est sauvegardé immédiatement ; relance le traitement pour reprendre après une interruption. La forme du sujet, sa structure corporelle et ses ailes visibles sont des dimensions séparées : un humanoïde peut donc aussi être bipède et ailé. Les annotations sont automatiques et demandent une validation humaine.

**Parcours :** configurer les chemins, charger le modèle, annoter, inspecter, puis exécuter la cellule de synchronisation si souhaité. Le modèle tourne sur la machine distante ; les images sont envoyées par un tunnel SSH. Le traitement peut demander beaucoup de temps.


## Exemple de résultat JSON

Une ligne JSONL correspond à une image. `labels` contient les feuilles choisies par l’IA et `taxonomy_paths` contient leurs chemins complets, reconstruits et vérifiés à partir de `taxonomies.json`.

```json
{
  "source": {"root": "data", "relative_path": "jeu/personnage.png"},
  "labels": {"asset_type": "single_sprite", "subject_type": "human_like", "view": "front", "pose": "standing", "action": "idle", "background_type": "transparent", "render_family": "pixel_art", "body_plan": "biped", "wings": "wings_not_visible", "background_scene": "not_applicable", "visual_quality": "unknown"},
  "taxonomy_paths": {"asset_type": ["sprite", "single_sprite"], "subject_type": ["animate_figure", "humanoid", "human_like"], "view": ["subject_angle", "front"], "pose": ["grounded", "standing"], "action": ["stationary", "idle"], "background_type": ["transparent"], "render_family": ["pixel_art"], "body_plan": ["legged", "biped"], "wings": ["wings_not_visible"], "background_scene": ["not_applicable"], "visual_quality": ["unknown"]},
  "caption_candidate": "A front-facing pixel-art character standing still.",
  "need_rework": null,
  "review_status": "auto_labeled",
  "needs_taxonomy_review": true
}
```


## 1. Dépendances

Le notebook s’exécute localement ; le modèle et le GPU restent sur le serveur distant.


In [ ]:
%pip install -q huggingface_hub pillow tqdm requests python-dotenv


## 2. Configuration et fichiers

Le dataset local provient du téléchargement du viewer. `MAX_IMAGES = None` traite toutes les images et planches ; `INCLUDE_SOURCE_SHEETS = False` limite le traitement à `data/` ; commence par un petit nombre pour vérifier les résultats. Les images Git LFS absentes sont téléchargées au besoin depuis la révision locale du dataset. Le jeton Hugging Face est lu depuis `HF_TOKEN`, un fichier `.env` local ou la connexion `hf auth login` ; il doit avoir accès en écriture pour la synchronisation.


In [ ]:
import json, os, hashlib, time, base64, io, subprocess
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
import requests
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from pathlib import Path
from datetime import datetime, timezone
from tqdm.auto import tqdm
from PIL import Image

cwd = Path.cwd().resolve()
DATASET_NOTEBOOK = next((p for p in (cwd, cwd / "notebook/dataset", cwd / "dataset") if (p / "taxonomies.json").is_file()), None)
if DATASET_NOTEBOOK is None:
    raise FileNotFoundError("Ouvre le notebook depuis notebook/dataset, notebook ou la racine du projet.")
PROJECT_ROOT = DATASET_NOTEBOOK.parent.parent
DATASET_ROOT = PROJECT_ROOT / "dataset-viewer/datasets/pixelart-dataset/.gitrepo"
TAXONOMY_PATH = DATASET_NOTEBOOK / "taxonomies.json"
OUTPUT_PATH = DATASET_NOTEBOOK / "taxonomy_work/annotations_v3_3_strata_gyro_s.jsonl"
MODEL_ID = None  # Détecté sur le serveur Strata dans la cellule suivante.
STRATA_URL = "http://127.0.0.1:8080"  # Tunnel SSH local vers Strata.
CONCURRENCY = 2  # Ajuster après mesure selon la VRAM et le débit.
REQUEST_TIMEOUT = 300
REPO_ID = "ethanzxv/pixelart-dataset"
# Ne pas écrire le jeton dans le notebook : variable d’environnement, connexion HF ou fichier .env local.
for env_file in (DATASET_NOTEBOOK / ".env", DATASET_NOTEBOOK.parent / ".env", Path.home() / "iris-sprite-lora/training/.env"):
    if env_file.is_file():
        load_dotenv(env_file)
        break
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF_ACCESS_TOKEN") or None
try:
    DATASET_REVISION = subprocess.run(["git", "-C", str(DATASET_ROOT), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    raise RuntimeError("Impossible de déterminer la révision de la copie locale du dataset.")
MAX_IMAGES = 10  # Mettre None pour traiter tout le dataset.
INCLUDE_SOURCE_SHEETS = True  # Inclut les planches originales en plus de data/.

if not (DATASET_ROOT / "data").is_dir():
    raise FileNotFoundError(f"Dataset local introuvable : {DATASET_ROOT / 'data'}. Télécharge-le depuis le viewer.")
taxonomy = json.loads(TAXONOMY_PATH.read_text(encoding="utf-8"))
def build_leaf_paths(tree, prefix=()):
    if not isinstance(tree, dict) or not tree:
        raise ValueError("Chaque hiérarchie doit être un arbre non vide.")
    paths = {}
    for name, children in tree.items():
        if not isinstance(name, str) or not name or not isinstance(children, dict):
            raise ValueError("Les nœuds doivent être des noms non vides associés à des objets.")
        path = (*prefix, name)
        if children:
            nested = build_leaf_paths(children, path)
            if paths.keys() & nested.keys():
                raise ValueError("Une feuille doit être unique dans sa dimension.")
            paths.update(nested)
        else:
            if name in paths:
                raise ValueError(f"Feuille dupliquée : {name}")
            paths[name] = list(path)
    return paths

hierarchies = taxonomy["hierarchies"]
leaf_paths = {dimension: build_leaf_paths(tree) for dimension, tree in hierarchies.items()}
fields = {dimension: list(paths) for dimension, paths in leaf_paths.items()}
def all_nodes(tree):
    return {name for name, children in tree.items()} | {name for children in tree.values() for name in all_nodes(children)}
value_descriptions = taxonomy["value_descriptions"]
if set(value_descriptions) != set(hierarchies):
    raise ValueError("Chaque dimension doit avoir ses descriptions de valeurs.")
for dimension, tree in hierarchies.items():
    if set(value_descriptions[dimension]) != all_nodes(tree) or not all(isinstance(s, str) and s.strip() for s in value_descriptions[dimension].values()):
        raise ValueError(f"Descriptions absentes ou supplémentaires pour {dimension}.")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
roots = ["data"] + (["source_sheets"] if INCLUDE_SOURCE_SHEETS and (DATASET_ROOT / "source_sheets").is_dir() else [])
images = [(root, p) for root in roots for p in sorted((DATASET_ROOT / root).rglob("*")) if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".gif"}]
if not images:
    raise FileNotFoundError("Aucune image trouvée dans le dataset local.")
def is_lfs_pointer(path):
    with path.open("rb") as f:
        return f.read(42).startswith(b"version https://git-lfs.github.com/spec/v1")
if all(is_lfs_pointer(path) for _, path in images[:min(32, len(images))]):
    print("Les images Git LFS manquantes seront téléchargées à la demande depuis la révision locale du dataset.")
print(f"{len(images)} images trouvées. Résultats : {OUTPUT_PATH}")
print("Champs :", ", ".join(fields))


## 3. Connecter le serveur Strata

Le serveur utilise Qwen3.8-Flash-Next Gyro-S quantifié avec les images activées. Sur la machine distante, il peut être démarré par :

```bash
~/.local/bin/strata launch --family gyro --model gyro-s --context 8192 --vision cpu --parallel 2 --drafting off --tune skip --access local --port 8080 --no-tui
```

Sur la machine où tourne ce notebook, ouvre un terminal et maintiens ce tunnel SSH actif :

```bash
ssh -N -L 8080:127.0.0.1:8080 -p 20395 eth@213.44.142.251
```

Ne mets pas de mot de passe dans le notebook. Le serveur reste accessible uniquement sur la machine distante et par le tunnel. Avec 16 Go de VRAM, la parallélisation peut réduire la mémoire disponible pour le modèle : comparer les débits à une et deux requêtes.


In [ ]:
response = requests.get(f"{STRATA_URL}/v1/models", timeout=10)
response.raise_for_status()
available = [item["id"] for item in response.json()["data"]]
if len(available) != 1:
    raise RuntimeError(f"Attendu un modèle Strata actif, trouvé : {available}")
MODEL_ID = available[0]
print(f"Serveur prêt : {MODEL_ID} · {STRATA_URL}")


## 4. Annoter et reprendre

Les requêtes sont envoyées en parallèle et la progression affiche une estimation basée sur le débit mesuré. Le notebook n’écrase pas les annotations existantes du dataset. Les lignes déjà présentes dans le fichier de travail sont conservées ; seules les images manquantes sont traitées. La qualité visuelle produit aussi `need_rework` : `true` si une retouche est proposée, `false` si aucun défaut majeur n’est détecté, `null` en cas de doute. Cette décision reste automatique et doit être vérifiée. Les erreurs sont affichées et réessayées à la prochaine exécution.


In [ ]:
def read_records(path):
    records = {}
    if path.exists():
        with path.open(encoding="utf-8") as f:
            for number, line in enumerate(f, 1):
                try:
                    row = json.loads(line)
                    key = (row["source"]["root"], row["source"]["relative_path"])
                except (ValueError, KeyError, TypeError) as exc:
                    raise ValueError(f"Ligne {number} invalide dans {path}; corrige-la avant de reprendre.") from exc
                if key in records:
                    raise ValueError(f"Doublon dans les résultats : {key}")
                records[key] = row
    return records

def validate_response(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    obj = json.loads(raw)
    labels = obj["labels"]
    if not isinstance(labels, dict) or set(labels) != set(fields):
        raise ValueError("Les champs de labels ne correspondent pas à la taxonomie.")
    for key, choices in fields.items():
        if labels[key] not in choices:
            raise ValueError(f"Valeur invalide pour {key}: {labels[key]}")
    if labels["background_type"] in {"transparent", "solid_color"} and labels["background_scene"] != "not_applicable":
        raise ValueError("Un fond transparent ou uni ne peut pas avoir de décor complexe.")
    if labels["background_type"] == "complex" and labels["background_scene"] == "not_applicable":
        raise ValueError("Un fond complexe doit avoir un décor ou unknown.")
    caption = obj.get("caption_candidate")
    if not isinstance(caption, str) or not caption.strip():
        raise ValueError("Description absente.")
    return labels, caption.strip()

def annotate(path):
    if is_lfs_pointer(path):
        relative_hub_path = path.relative_to(DATASET_ROOT).as_posix()
        path = Path(hf_hub_download(REPO_ID, relative_hub_path, repo_type="dataset", revision=DATASET_REVISION, token=HF_TOKEN))
    instruction = ("Analyze only visible pixels. Return ONLY a JSON object with keys labels and caption_candidate. "
                   "labels must contain exactly these dimensions; choose one leaf name (an entry with an empty object) per dimension from this hierarchy: "
                   + json.dumps(hierarchies, ensure_ascii=False) +
                   " Dimension definitions and applicability: " + json.dumps(taxonomy["dimensions"], ensure_ascii=False) +
                   " Definitions of each branch and possible leaf: " + json.dumps(value_descriptions, ensure_ascii=False) +
                   " Allowed leaf values for each field: " + json.dumps(fields, ensure_ascii=False) +
                   " For each field choose exactly one of its own allowed values. Do not return branch names or values from other fields. If background_type is transparent or solid_color, background_scene must be not_applicable; if background_type is complex, background_scene must be a scene or unknown. Use unknown when unsure, not_applicable when irrelevant. Caption in English, concise and visual; do not infer game roles or rights.")
    with Image.open(path) as source:
        image = source.convert("RGBA")
        has_transparency = image.getchannel("A").getextrema()[0] < 255
        instruction += " The original image contains transparent pixels." if has_transparency else " The original image has no transparent pixels."
        scale = max(1, min(8, 512 // max(image.size)))
        image = image.resize((image.width * scale, image.height * scale), Image.Resampling.NEAREST)
        background = Image.new("RGBA", image.size, "white")
        background.alpha_composite(image)
        buffer = io.BytesIO()
        background.convert("RGB").save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    body = {"model": MODEL_ID, "messages": [{"role": "user", "content": [
        {"type": "text", "text": instruction},
        {"type": "image_url", "image_url": {"url": "data:image/png;base64," + encoded}},
    ]}], "temperature": 0, "max_tokens": 768, "response_format": {"type": "json_object"}, "chat_template_kwargs": {"enable_thinking": False}}
    for attempt in range(2):
        response = requests.post(f"{STRATA_URL}/v1/chat/completions", json=body, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"]
        try:
            return validate_response(content)
        except (ValueError, KeyError, TypeError) as error:
            if attempt:
                raise
            body["messages"].extend([
                {"role": "assistant", "content": content},
                {"role": "user", "content": f"Invalid taxonomy response: {error}. Return a corrected JSON object. Use only the allowed leaf values for each field; use unknown if unsure."},
            ])

records = read_records(OUTPUT_PATH)
taxonomy_hash = hashlib.sha256(TAXONOMY_PATH.read_bytes()).hexdigest()
if any(row.get("taxonomy_sha256") != taxonomy_hash for row in records.values()):
    raise ValueError("La taxonomie a changé depuis ce lot ; choisis un nouveau OUTPUT_PATH pour éviter de mélanger les versions.")
pending = [(root, p) for root, p in images if (root, p.relative_to(DATASET_ROOT / root).as_posix()) not in records]
if MAX_IMAGES is not None:
    pending = pending[:MAX_IMAGES]
if CONCURRENCY < 1:
    raise ValueError("CONCURRENCY doit être supérieur à zéro.")
errors = []
started = time.monotonic()
completed = 0
with ThreadPoolExecutor(max_workers=CONCURRENCY) as executor, tqdm(total=len(pending), desc="Annotation", unit="image") as progress:
    # Maintain a bounded queue and refill as soon as one request finishes.
    pending_iter = iter(pending)
    futures = {}
    def submit_next():
        item = next(pending_iter, None)
        if item is not None:
            root, path = item
            futures[executor.submit(annotate, path)] = (root, path)
    for _ in range(CONCURRENCY):
        submit_next()
    consecutive_errors = 0
    while futures:
        done, _ = wait(futures, return_when=FIRST_COMPLETED)
        for future in done:
            root, path = futures.pop(future)
            relative = path.relative_to(DATASET_ROOT / root).as_posix()
            try:
                labels, caption = future.result()
                row = {"asset_id": hashlib.sha256((root + "/" + relative).encode()).hexdigest()[:24],
                       "source": {"root": root, "relative_path": relative}, "labels": labels,
                       "taxonomy_paths": {key: leaf_paths[key][value] for key, value in labels.items()},
                       "caption_candidate": caption, "caption_status": "candidate_unreviewed",
                       "need_rework": None if labels["visual_quality"] == "unknown" else labels["visual_quality"] == "needs_rework",
                       "review_status": "auto_labeled", "needs_taxonomy_review": True,
                       "rights_status": "unknown", "model": MODEL_ID, "taxonomy_version": taxonomy["version"],
                       "taxonomy_sha256": taxonomy_hash, "annotated_at_utc": datetime.now(timezone.utc).isoformat()}
                with OUTPUT_PATH.open("a", encoding="utf-8") as f:
                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
                records[(root, relative)] = row
                completed += 1
                consecutive_errors = 0
            except Exception as exc:
                errors.append((relative, str(exc)))
                consecutive_errors += 1
            progress.update(1)
            if completed:
                rate = completed / (time.monotonic() - started)
                hours = max(0, len(images) - len(records)) / rate / 3600
                progress.set_postfix_str(f"{rate:.2f} images/s · reste ~{hours:.1f} h (hors erreurs)")
            if consecutive_errors >= 10:
                raise RuntimeError(f"10 erreurs successives ; traitement interrompu. Dernière erreur : {errors[-1][1]}")
            submit_next()

print(f"{len(records)} annotations sauvegardées ; {len(errors)} erreurs ; {len(images) - len(records)} images restantes.")
for path, error in errors[:10]:
    print(f"- {path}: {error}")


## 5. Vérifier un exemple

Les étiquettes restent des propositions. Vérifie les images et les droits avant toute utilisation pour l'entraînement.


In [ ]:
from IPython.display import display
records = read_records(OUTPUT_PATH)
if records:
    example = next(iter(records.values()))
    display(Image.open(DATASET_ROOT / example["source"]["root"] / example["source"]["relative_path"]))
    print(json.dumps(example, indent=2, ensure_ascii=False))
else:
    print("Aucune annotation disponible ; exécute la cellule précédente.")


## 6. Synchroniser sur Hugging Face

Exécute cette cellule quand tu veux publier l'état actuel. Elle envoie toutes les annotations produites dans le fichier de travail et la taxonomie ensemble dans un nouveau dossier horodaté sous `annotations/taxonomy_runs/` ; elle ne supprime ni ne remplace les fichiers existants. Les annotations sont marquées comme non vérifiées. Le jeton doit disposer des droits d'écriture sur le dataset.


In [ ]:
from huggingface_hub import HfApi, CommitOperationAdd

records = read_records(OUTPUT_PATH)
if not records:
    raise ValueError("Aucune annotation à synchroniser.")
current_hash = hashlib.sha256(TAXONOMY_PATH.read_bytes()).hexdigest()
if any(row.get("taxonomy_sha256") != current_hash for row in records.values()):
    raise ValueError("La taxonomie ne correspond pas aux annotations : utilise le fichier de taxonomie de ce lot.")
api = HfApi()
revision = api.repo_info(REPO_ID, repo_type="dataset").sha
run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
remote_dir = f"annotations/taxonomy_runs/{run_name}"
commit = api.create_commit(
    repo_id=REPO_ID, repo_type="dataset", parent_commit=revision,
    operations=[CommitOperationAdd(path_in_repo=f"{remote_dir}/annotations.jsonl", path_or_fileobj=str(OUTPUT_PATH)),
                CommitOperationAdd(path_in_repo=f"{remote_dir}/taxonomies.json", path_or_fileobj=str(TAXONOMY_PATH))],
    commit_message=f"Add taxonomy annotations ({len(records)} images)",
)
print(f"{len(records)} annotations synchronisées dans {REPO_ID}/{remote_dir}")
print(commit.commit_url)
